# Ridge Regression — L2 Regularization

## 1. Definition and intuition
Ridge regression fits a linear model while penalizing large squared coefficients. It is especially useful when features are correlated or the model has many weakly informative predictors.

## 2. Objective
\\[\\min_{w,b}\\; \\|y-Xw-b\\mathbf{1}\\|_2^2 + \\alpha\\|w\\|_2^2.\\]

## 3. Why it works: derivation and assumptions
Center \\(X\\) and \\(y\\) so the intercept is handled separately. Differentiating with respect to \\(w\\) gives \\(-2X^T(y-Xw)+2\\alpha w=0\\), hence \\((X^TX+\\alpha I)w=X^Ty\\). For \\(\\alpha>0\\), the matrix is positive definite, so the solution is unique even when columns of \\(X\\) are linearly dependent.

**Assumptions and failure modes:** The linear prediction form is useful, but coefficients may be biased toward zero. L2 regularization stabilizes the solution; it does not perform exact feature selection. Feature scaling matters because the penalty acts on coefficient magnitudes.

## 4. Dataset and preprocessing

We use a synthetic regression task with informative features and correlated columns. Predictors are standardized using statistics fit on the training split only. This prevents test-set leakage and makes coefficient penalties comparable across feature scales.

In [1]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
rng = np.random.default_rng(SEED)
X, y, true_coef = make_regression(
    n_samples=700, n_features=24, n_informative=8,
    noise=12.0, coef=True, random_state=SEED
)
# Add correlated columns to make the regularization trade-off visible.
X[:, 20] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X[:, 21] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (525, 24) Test: (175, 24)


## 5. From-scratch implementation

The implementation centers the training data to estimate an unpenalized intercept, then fits coefficients using the objective defined above. It validates shapes and parameter ranges.

In [2]:
def fit_ridge(X, y, alpha=5.0):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
        raise ValueError("X must be non-empty 2D and y must be matching 1D")
    if alpha < 0:
        raise ValueError("alpha must be nonnegative")
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - x_mean, y - y_mean
    A = Xc.T @ Xc + alpha * np.eye(X.shape[1])
    coef = np.linalg.solve(A, Xc.T @ yc)
    intercept = y_mean - x_mean @ coef
    return intercept, coef

def predict_linear(X, intercept, coef):
    return np.asarray(X) @ coef + intercept

alpha = 5.0
b_scratch, w_scratch = fit_ridge(X_train, y_train, alpha)
pred_scratch = predict_linear(X_test, b_scratch, w_scratch)
print("From-scratch MSE:", mean_squared_error(y_test, pred_scratch))
print("From-scratch R²:", r2_score(y_test, pred_scratch))


From-scratch MSE: 167.4815525598888
From-scratch R²: 0.9886951915889054


## 6. Compare with scikit-learn

Compare predictions, not just coefficient vectors. Coefficients can be affected by solver tolerance and collinearity. The library and scratch implementations below use matching objective conventions.

In [3]:
model = Ridge(alpha=alpha).fit(X_train, y_train)
pred_sklearn = model.predict(X_test)
print("scikit-learn MSE:", mean_squared_error(y_test, pred_sklearn))
print("scikit-learn R²:", r2_score(y_test, pred_sklearn))
print("Max prediction difference:", np.max(np.abs(pred_scratch - pred_sklearn)))
assert np.allclose(pred_scratch, pred_sklearn, atol=1e-6)


scikit-learn MSE: 167.4815525598887
scikit-learn R²: 0.9886951915889054
Max prediction difference: 2.2737367544323206e-13


## 7. Hyperparameters

`alpha` controls shrinkage. Larger values reduce coefficient magnitudes and variance but can increase bias. Tune it with cross-validation inside the training data; do not select it on the final test set.

## 8. Best practices

Standardize predictors; preserve the intercept outside the penalty; consider sparse solvers for sparse features; inspect train/validation curves; compare with OLS and Lasso; avoid explicitly inverting matrices.

## 9. Time/space complexity

For dense data with n rows and d features, the closed-form solve costs roughly O(nd² + d³) when n ≥ d and forms d×d matrices. Batch inference costs O(md), model storage O(d). For d≫n, use a dual formulation or iterative solver rather than forming a d×d matrix.

## 10. Scaling to millions of samples

Use float32 when accuracy allows, sparse-aware solvers for sparse matrices, and iterative/minibatch methods for large n. Avoid materializing copies unnecessarily. For very high dimensions, choose solvers based on the relationship between n and d. Benchmark quality, runtime and peak memory together.

## 11. GPU suitability

A GPU is most promising when the implementation is dominated by large dense matrix operations. Coordinate descent updates one coefficient at a time, so it may be less GPU-friendly than proximal-gradient optimization. Always include data transfer and device memory in timing.

In [4]:
# Optional PyTorch GPU solve. A GPU helps only if matrices are large enough to offset transfer overhead.
try:
    import torch
    if torch.cuda.is_available():
        device = torch.device("cuda")
        Xt = torch.as_tensor(X_train, dtype=torch.float32, device=device)
        yt = torch.as_tensor(y_train, dtype=torch.float32, device=device)
        xm, ym = Xt.mean(0), yt.mean()
        Xc, yc = Xt-xm, yt-ym
        wt = torch.linalg.solve(Xc.T @ Xc + alpha * torch.eye(Xt.shape[1], device=device), Xc.T @ yc)
        bt = ym - xm @ wt
        print("GPU Ridge test MSE:", mean_squared_error(y_test, (torch.as_tensor(X_test, dtype=torch.float32, device=device) @ wt + bt).cpu().numpy()))
    else:
        print("CUDA unavailable; GPU cell skipped. The CPU implementation above remains usable.")
except ImportError:
    print("PyTorch is optional and is not installed in this environment.")


CUDA unavailable; GPU cell skipped. The CPU implementation above remains usable.


## 12. Correctness tests and synthesis

The regularizer trades coefficient flexibility for stability. Ridge shrinks coefficients continuously, Lasso can make them exactly zero, and Elastic Net interpolates between these behaviors. The validation strategy matters as much as the optimizer.

In [5]:
assert np.isfinite(pred_scratch).all()
assert len(pred_scratch) == len(y_test)
assert mean_squared_error(y_test, pred_scratch) < np.var(y_test)
try:
    fit_ridge(np.array([1, 2, 3]), np.array([1, 2, 3]), 1.0)
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid X dimensionality to raise")
print("All Ridge checks passed.")


All Ridge checks passed.
